# Synergy model interpretation for individual drug combinations

## Data loading

Configure root with local/colab.

In [ ]:
import sys, subprocess
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
from pathlib import Path

# Configure root
COLAB = Path("/content").exists()
repo_url = "https://github.com/eddykang06/phenotype-prediction.git"
repo_dir = Path("phenotype-prediction")
if COLAB:
    root = Path("/content/phenotype-prediction")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", repo_url])
else:
    root = Path.cwd().parent
sys.path.insert(0, str(root))

Configure data path for colab/local.

In [ ]:
if COLAB:
  from google.colab import drive
  drive.mount("/content/drive")
  data_dir = Path("/content/drive/MyDrive/phenotype-prediction-data")
  l2fc_dir = str(data_dir / "dge")
  cfu_dir = str(data_dir /  "cfus")
  annot_path = str(data_dir / "Annotation_TIGR4.tsv")

else:
  data_dir = Path("C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab")
  l2fc_dir = str(data_dir / "dge")
  cfu_dir = str(data_dir / "cfus")
  annot_path = str(data_dir / "Annotation_TIGR4.tsv")

Load the time-matched transcriptional scores, synergy scores, and metadata.

In [ ]:
from src.dge_data import (
    simple_interaction_score,
    eob_score,
    get_all_synergy_data
)

# Bliss score and simple interaction score
data_df = get_all_synergy_data(
    l2fc_dir = l2fc_dir,
    cfu_dir = cfu_dir,
    interaction_score_method = simple_interaction_score,
    synergy_score_method = eob_score,
    time_matched = True
)

# Drop genes with NA values
data_df = data_df.dropna(axis = 1)

# Load annotations
annotations = pd.read_table(annot_path, sep = "\t")
annotations.set_index("TIGR4.old", inplace = True, drop = True)

## CEF+CIP feature interpretation

Feature importance strategy:
- Train 5-fold CV 
- Find features with consistently high coefficients (mean / std) -> candidate mechanistic contributors to prediction
- Find expression levels of these genes in the original data (heatmap)
- Once candidates identified, use statistical testing in low and high synergy groups?

Train and extract features.

In [ ]:
from sklearn.model_selection import KFold
from src.train import run_nested_pls_cv
from src.interpret import cv_feature_importances, plot_top_features

# Random splits
cv = KFold(
    n_splits = 5,
    shuffle = True,
    random_state = 111
)

# Isolate cefcip data
cefcip_df = data_df[data_df["drug_id"] == "CEF+CIP"]

# Get splits
cefcip_splits = list(cv.split(cefcip_df))

# Get performance
cefcip_scores = run_nested_pls_cv(
        df = cefcip_df,
        splits = cefcip_splits,
        synergy = True
)

# Get feature importances
cefcip_features = cv_feature_importances(
    df = cefcip_df,
    splits = cefcip_splits
)

# Plot top features
plot_top_features(
    coef_df = cefcip_features,
    annot_df = annotations,
    top_n = 40,
    xlabel = "Coefficient mean/std over 5 folds",
    title = "Top 40 features predictive of CEF+CIP synergy"   
)

Heatmap of top predictive genes' interaction scores.

In [ ]:
# Get dataframe ordered by feature importances
ordered_cefcip = cefcip_df[cefcip_features.index.to_list() + ["synergy_score"]]

plt.figure(figsize = (15, 10))
sns.heatmap(ordered_cefcip.T[:50], cmap = "coolwarm")
plt.title("Heatmap of interaction scores for top 50 genes predictive of CEF+CIP synergy")
plt.tight_layout()

Run Mann-Whitney + FDR on top 100 predictive genes, splitting groups as high and low synergy.

In [ ]:
from scipy.stats import mannwhitneyu, false_discovery_control

# Add column of high and low synergy
ordered_cefcip["group"] = ordered_cefcip["synergy_score"].transform(lambda x: "high" if x > 0 else "low")

# Run welch t-test for each 
gene_cols = ordered_cefcip.columns[ordered_cefcip.columns.str.contains("SP")][:100]

# Store results
results = []

# Presplit the groups and use only top 100 predictive gnes
high = ordered_cefcip[ordered_cefcip["group"] == "high"]
low = ordered_cefcip[ordered_cefcip["group"] == "low"]

for gene in gene_cols:
    x = high[gene]
    y = low[gene]

    # Mann whitney 
    u, p = mannwhitneyu(x, y)
    results.append({
        "gene": gene,
        "p_value": p
    })

results = pd.DataFrame(results)

# FDR
results["fdr"] = false_discovery_control(results["p_value"], method = "bh")
results

## CIP+VNC feature interpretation

In [ ]:
# Isolate data
cipvnc_df = data_df[data_df["drug_id"] == "CIP+VNC"]

# Get splits
cipvnc_splits = list(cv.split(cipvnc_df))

# Get performance
cipvnc_scores = run_nested_pls_cv(
        df = cipvnc_df,
        splits = cipvnc_splits,
        synergy = True
)

# Get feature importances
cipvnc_features = cv_feature_importances(
    df = cipvnc_df,
    splits = cipvnc_splits
)

# Plot top features
plot_top_features(
    coef_df = cipvnc_features,
    annot_df = annotations,
    top_n = 50,
    xlabel = "Coefficient mean/std over 5 folds",
    title = "Top 30 features for CIP+VNC regression model"   
)

## CEF+RIF feature interpretation

In [ ]:
# Isolate cefrif data
cefrif_df = data_df[data_df["drug_id"] == "CEF+RIF"]

# Get splits
cefrif_splits = list(cv.split(cefrif_df))

# Get performance
cefrif_scores = run_nested_pls_cv(
        df = cefrif_df,
        splits = cefrif_splits,
        synergy = True
)

# Get feature importances
cefrif_features = cv_feature_importances(
    df = cefrif_df,
    splits = cefrif_splits
)

# Plot top features
plot_top_features(
    coef_df = cefrif_features,
    annot_df = annotations,
    top_n = 30,
    xlabel = "Coefficient mean/std over 5 folds",
    title = "Top 30 features for CEF+RIF regression model"   
)